# Quantized Retail Recommender

This notebook runs the complete synthetic retail recommendation experiment. It trains a compact causal Transformer **from scratch** (not continued pretraining), applies dynamic and calibrated static INT8 post-training quantization, and compares all variants on the same held-out split. In Colab, select a CPU or GPU runtime, upload/clone this project, and set the working directory to the project root.

In [ ]:
# Install project dependencies in a Colab runtime if they are not installed yet.
%pip install -q -r requirements.txt

In [ ]:
from pathlib import Path
import torch

ROOT = Path.cwd()
assert (ROOT / "src" / "pipeline.py").exists(), "Change directory to the project root first."
print("PyTorch:", torch.__version__)
print("Training device:", "cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
from src.pipeline import prepare_data
from src.dataset import load_transaction_baskets, split_transactions, ProductTokenizer

tokenizer = prepare_data(transaction_count=10_000)
baskets = load_transaction_baskets()
train_transactions, validation_transactions, test_transactions = split_transactions(baskets)
print(f"Transactions: {len(baskets):,}; products: {len(tokenizer.products)}")
print(f"Train/validation/test: {len(train_transactions):,}/{len(validation_transactions):,}/{len(test_transactions):,}")

In [ ]:
from src.train import train_model

baseline, history = train_model(
    tokenizer=tokenizer,
    train_transactions=train_transactions,
    validation_transactions=validation_transactions,
    test_transactions=test_transactions,
    epochs=4,
)
print("Training loss:", history["train_loss"])
print("Validation loss:", history["validation_loss"])

In [ ]:
from src.dataset import build_next_item_examples
from src.quantization import evaluate_quantized_models
from src.evaluate import build_comparison

calibration_examples = build_next_item_examples(
    train_transactions, tokenizer, max_length=baseline.max_length, shuffle_baskets=True
)
metrics = evaluate_quantized_models(baseline, test_transactions, tokenizer, calibration_examples)
comparison = build_comparison(metrics)
comparison

In [ ]:
from src.recommend import recommend_next, probe_itemsets

basket = ["Milk", "Bread", "Butter"]
print("Model recommendations:", recommend_next(basket, model=baseline, tokenizer=tokenizer, top_k=3))
print("Observed itemset associations:", probe_itemsets(basket, transactions=baskets, top_k=5))

The pipeline saves actual checkpoint sizes and held-out results under `models/` and `results/`. Static quantization calibrates supported Linear layers; embeddings, normalization, dropout, and attention matrix operations remain floating point. Inspect the saved comparison CSV and plots rather than assuming quantization improves every metric. In Colab, download needed outputs from the Files panel or package them before the runtime ends.